# RS-Code-SSM — Full training pipeline (PAPER §4)

**Before running:**
1. **Secrets →** add **`HF_TOKEN`** (read access to your dataset + model repos), **or**
2. **Add Data →** attach a Kaggle dataset / file that contains **`all_traces.jsonl`** (copied automatically from `/kaggle/input`).


This single notebook mirrors **PAPER.md §4** on a **Kaggle GPU** (or any machine with the repo + data).

| Stage | Paper | In this notebook |
|-------|--------|-------------------|
| **Optional pretrain** | The Stack / code LM (not §4.1, but ROADMAP Phase 1) | **§ Optional A** — streams HF data; long run |
| **§4.1 EpiChat SFT** | Traces from EpiChat export | Use **`all_traces.jsonl`** from Hugging Face (includes merged EpiChat + supplements) |
| **§4.2 RFT** | Ollama DeepSeek-R1 + rejection sampling | **Not runnable on Kaggle** (no Ollama). Your HF dataset should already include verified / supplemental traces. Run `train.reasoning_data` locally if you need fresh RFT |
| **§4.3 GRPO** | Execution reward, 3000 steps | **§ GRPO** |
| **§4.4 Self-improve** | 3× merge → SFT → GRPO | **§ Optional C** — optional loops (short SFT+GRPO); full teacher regen needs local `pipeline_96.sh` |
| **Verifier (abstract)** | ~100M verifier | **§ Optional D** |

**Setup:** enable **`HF_TOKEN`** in Kaggle Secrets, **GPU T4**, **Internet**. Upload traces to dataset `pgalyen1987/rs-code-ssm-traces` (`all_traces.jsonl` at minimum).

**Runtime:** optional pretrain ~hours; SFT ~45–90 min; GRPO ~6–9 h — may require **multiple sessions** (checkpoints resume).

In [ ]:
# ── CONFIG: paper-aligned defaults (edit here) ───────────────────────────────
import os

# Repos
REPO = 'pgalyen1987/RS-Code-SSM'
DATA_REPO = 'pgalyen1987/rs-code-ssm-traces'
HF_MODEL_REPO = 'pgalyen1987/RS-Code-SSM-1.6B'  # HF weights; GitHub REPO is RS-Code-SSM without suffix

# Optional Phase A: pretrain on The Stack (very long). False = skip (paper §4 assumes you may start from scratch or use init).
RUN_PRETRAIN = False
PRETRAIN_MAX_TOKENS = 50_000_000   # lower for smoke; paper uses 2e9 over many sessions

# After pretrain (or download), initialize SFT from this path if it exists
PRETRAIN_CKPT_GUESS = 'checkpoints/pretrain/pretrain_final.pt'  # or latest pretrain_*.pt

# SFT — PAPER §4.2 (merged traces): 3 epochs, batch 1, grad_accum 16, seq 1024
SFT_OUTPUT = 'checkpoints/sft_v2'
SFT_EPOCHS = 3
SFT_LR = 2e-4
SFT_GRAD_ACCUM = 16
SFT_MAX_SEQ = 1024

# GRPO — PAPER §4.3
GRPO_DIR = 'checkpoints/grpo'
GRPO_STEPS = 3000
GRPO_LR = 5e-6
GRPO_KL = 0.04
GRPO_GROUP = 8

# Optional §4.4: self-improve iterations (each: SFT 2 epochs + GRPO 2000 steps — paper suggests smaller LR)
SELF_IMPROVE_ITERS = 0  # set 1–3 only if you have time; uses same traces

# Optional verifier — PAPER §5.2
RUN_VERIFIER = False
VERIFIER_EPOCHS = 5
VERIFIER_DIR = 'checkpoints/verifier'

# Export
PUSH_TO_HF = True
EXPORT_VERSION = '0.1.0'

print('CONFIG OK')

In [ ]:
# ── Install dependencies (Kaggle-friendly) ───────────────────────────────────
import os, subprocess, sys

# Quiet debugger / papermill noise (set before any other imports that might hook Python)
os.environ.setdefault('PYDEVD_DISABLE_FILE_VALIDATION', '1')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch

def pip(*args, timeout=900):
    try:
        r = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q', *args],
            capture_output=True, text=True, timeout=timeout,
        )
    except subprocess.TimeoutExpired:
        print('  TIMEOUT', args[0] if args else '')
        return False
    if r.returncode != 0:
        tail = (r.stderr or r.stdout or '')[-800:]
        print('  FAIL:', args[0] if args else '', '→', tail[-400:])
        return False
    print('  OK', args[0] if args else 'pkgs')
    return True

# Base ML stack (matches existing notebooks)
pip('safetensors', 'transformers', 'datasets', 'huggingface_hub', 'faiss-cpu', 'sentence-transformers')
pip('ninja', 'packaging', 'wheel', 'setuptools')

# Mamba-2 is implemented in arch/mamba2.py (pure PyTorch). No mamba-ssm / causal-conv1d packages.
print(f'PyTorch {torch.__version__}, cuda={torch.version.cuda}')
print('Dependencies installed')


In [ ]:
# ── HuggingFace auth + clone repo ────────────────────────────────────────────
import os, subprocess, sys
from pathlib import Path

HF_TOKEN = ''
try:
    from kaggle_secrets import UserSecretsClient
    client = UserSecretsClient()
    for label in ('HF_TOKEN', 'hf_token', 'HUGGINGFACE_TOKEN', 'huggingface_token'):
        try:
            HF_TOKEN = client.get_secret(label)
            if HF_TOKEN:
                print(f'HF: secret {label!r}')
                break
        except Exception:
            pass
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)

if not Path('arch').exists():
    if not Path('RS-Code-SSM').exists():
        subprocess.run(
            ['git', 'clone', f'https://github.com/{REPO}.git', 'RS-Code-SSM'],
            check=True,
        )
    os.chdir('RS-Code-SSM')
    sys.path.insert(0, '.')
print('CWD:', Path.cwd())
if Path('.git').exists():
    _gp = subprocess.run(['git', 'pull', '--ff-only'], capture_output=True, text=True)
    print('git pull:', _gp.returncode, (_gp.stdout or _gp.stderr or '')[-300:].strip())

if HF_TOKEN:
    from ssm.hf_checkpoint_sync import download_checkpoint
    Path(SFT_OUTPUT).mkdir(parents=True, exist_ok=True)
    Path(GRPO_DIR).mkdir(parents=True, exist_ok=True)
    download_checkpoint(HF_MODEL_REPO, 'training/sft_latest.pt', Path(SFT_OUTPUT) / 'sft_resume_hf.pt', HF_TOKEN)
    download_checkpoint(HF_MODEL_REPO, 'training/grpo_latest.pt', Path(GRPO_DIR) / 'grpo_resume_hf.pt', HF_TOKEN)

In [ ]:
# ── GPU ──────────────────────────────────────────────────────────────────────
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {p.name}, {p.total_memory/1e9:.1f} GB')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# ── Download traces + optional pretrain checkpoint ───────────────────────────
from pathlib import Path
import shutil

Path('data').mkdir(parents=True, exist_ok=True)

def find_traces_in_kaggle_input():
    root = Path('/kaggle/input')
    if not root.is_dir():
        return None
    for p in root.rglob('all_traces.jsonl'):
        if p.is_file() and p.stat().st_size > 1000:
            return p
    return None

src = find_traces_in_kaggle_input()
if src is not None and not Path('data/all_traces.jsonl').exists():
    shutil.copy2(src, Path('data/all_traces.jsonl'))
    print('Using all_traces.jsonl from Kaggle input:', src)

if HF_TOKEN:
    from huggingface_hub import hf_hub_download
    if not Path('data/all_traces.jsonl').exists():
        try:
            hf_hub_download(
                repo_id=DATA_REPO, filename='all_traces.jsonl',
                repo_type='dataset', token=HF_TOKEN, local_dir='data',
            )
            print('Downloaded all_traces.jsonl from HF')
        except Exception as e:
            raise RuntimeError(
                f'No all_traces.jsonl locally and HF download failed: {e}\\n'
                'Add HF_TOKEN or Add Data with all_traces.jsonl'
            ) from e
    for fname in ('public_supplement.jsonl', 'kaggle_supplement.jsonl'):
        dest = Path('data') / fname
        if dest.exists():
            continue
        try:
            hf_hub_download(
                repo_id=DATA_REPO, filename=fname,
                repo_type='dataset', token=HF_TOKEN, local_dir='data',
            )
            print('Downloaded', fname)
        except Exception as e:
            print(fname, 'optional:', e)
    Path('checkpoints/pretrain').mkdir(parents=True, exist_ok=True)
    # Same path as train.pretrain uploads (training/pretrain_latest.pt); legacy: pretrain_checkpoint.pt
    _down = False
    for _fn in ('training/pretrain_latest.pt', 'pretrain_checkpoint.pt'):
        try:
            hf_hub_download(
                repo_id=HF_MODEL_REPO, filename=_fn,
                repo_type='model', token=HF_TOKEN,
                local_dir='checkpoints/pretrain',
            )
            print('Downloaded', _fn, '(optional)')
            _down = True
            break
        except Exception:
            continue
    if not _down:
        print('No pretrain checkpoint on Hub (optional): tried training/pretrain_latest.pt and pretrain_checkpoint.pt')
elif not Path('data/all_traces.jsonl').exists():
    raise RuntimeError(
        'No all_traces.jsonl. Add Kaggle Secret HF_TOKEN (dataset read) '
        'or Add Data → dataset containing all_traces.jsonl under /kaggle/input'
    )


In [ ]:
# ── Build train_full.jsonl (dedupe: prefer single merged file) ───────────────
import shutil
from pathlib import Path

merged = Path('data/all_traces.jsonl')
if merged.exists():
    shutil.copy(merged, Path('data/train_full.jsonl'))
    print('train_full ← all_traces.jsonl (merged bundle)')
else:
    parts = [Path('data')/n for n in ('public_supplement.jsonl','kaggle_supplement.jsonl','epichat_traces.jsonl') if (Path('data')/n).exists()]
    if not parts:
        raise RuntimeError('No traces')
    if len(parts) == 1:
        shutil.copy(parts[0], Path('data/train_full.jsonl'))
    else:
        with open('data/train_full.jsonl','w',encoding='utf-8') as out:
            for p in parts:
                t = p.read_text(encoding='utf-8')
                out.write(t)
                if t and not t.endswith('\n'):
                    out.write('\n')
    print('train_full from shards:', [p.name for p in parts])

print('lines:', sum(1 for _ in open('data/train_full.jsonl')))

In [ ]:
# ── OPTIONAL A: Pretrain (PAPER / ROADMAP Phase 1) ───────────────────────────
# Streams The Stack; skip if RUN_PRETRAIN is False
import subprocess, sys, os
from pathlib import Path

if not RUN_PRETRAIN:
    print('SKIP pretrain (RUN_PRETRAIN=False)')
else:
    from ssm.pretrain_checkpoint import find_best_pretrain_checkpoint
    cdir = Path('checkpoints/pretrain')
    rp, td, sd = find_best_pretrain_checkpoint(cdir)
    resume = None
    if rp is not None and not (td == 0 and sd == 0):
        resume = str(rp)
        print('Resume pretrain from', resume, f'(step={sd}, tokens={td:,})')
    elif rp is not None:
        print('Best checkpoint has step=0, tokens=0 — starting fresh (not using --resume)')
    cmd = [
        sys.executable, '-m', 'train.pretrain',
        '--output-dir', 'checkpoints/pretrain',
        '--model-size', '700m',
        '--max-tokens', str(PRETRAIN_MAX_TOKENS),
        '--batch-size', '1', '--grad-accum', '16', '--seq-len', '1024',
        '--lr', '1e-3', '--device', DEVICE,
        '--save-every', '5',
        '--hf-repo', HF_MODEL_REPO,
    ]
    if resume:
        cmd += ['--resume', resume]
    subprocess.run(cmd, check=True, env={**os.environ, 'HF_TOKEN': HF_TOKEN or '', 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'})

### §4.1–4.2 EpiChat + RFT data\n
\n
**Paper:** EpiChat traces + RFT (verified teacher traces) + optional CodeAlpaca merge.\n
**Here:** `train_full.jsonl` is your merged training file from Hugging Face. RFT regeneration requires **local Ollama** (`train.reasoning_data`); on Kaggle we only **train** on the uploaded bundle.\n

In [ ]:
# ── §4.2 / §4.3 — SFT (EpiChat+RFT+extras in one JSONL) ─────────────────────
import subprocess, sys, os
from pathlib import Path

traces = 'data/train_full.jsonl'
Path(SFT_OUTPUT).mkdir(parents=True, exist_ok=True)
SFT_DONE = Path(SFT_OUTPUT) / 'sft_final.pt'
if SFT_DONE.exists():
    print('SFT already complete:', SFT_DONE)
else:
    init_arg = []
    from ssm.pretrain_checkpoint import find_best_pretrain_checkpoint
    _pp, _pt, _ps = find_best_pretrain_checkpoint(Path('checkpoints/pretrain'))
    pp = Path(_pp) if _pp is not None else None
    if pp is not None and _pt == 0 and _ps == 0:
        print('[SFT] Pretrain file has no recorded progress; skipping --init-checkpoint')
        pp = None
    out = Path(SFT_OUTPUT)
    steps = sorted(out.glob('sft_step_*.pt'), key=lambda p: int(p.stem.split('_')[-1]))
    resume_ckpt = steps[-1] if steps else None
    if resume_ckpt is None and (out / 'sft_best.pt').exists():
        resume_ckpt = out / 'sft_best.pt'
    if resume_ckpt is None and (out / 'sft_resume_hf.pt').exists():
        resume_ckpt = out / 'sft_resume_hf.pt'
    if pp is not None and pp.exists() and resume_ckpt is None:
        init_arg = ['--init-checkpoint', str(pp)]
        print('SFT init from pretrain:', pp)
    cmd = [
        sys.executable, '-m', 'train.sft_reasoning',
        '--traces', traces,
        '--output-dir', SFT_OUTPUT,
        '--model-size', '700m',
        '--epochs', str(SFT_EPOCHS),
        '--lr', str(SFT_LR),
        '--batch-size', '1',
        '--grad-accum', str(SFT_GRAD_ACCUM),
        '--max-seq-len', str(SFT_MAX_SEQ),
        '--device', DEVICE,
    ] + init_arg
    if resume_ckpt is not None:
        cmd += ['--resume', str(resume_ckpt)]
        print('Resuming SFT from', resume_ckpt)
    subprocess.run(cmd, check=True, env={**os.environ, 'HF_TOKEN': HF_TOKEN or '', 'HF_MODEL_REPO': HF_MODEL_REPO, 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True', 'PYDEVD_DISABLE_FILE_VALIDATION': '1'})

SFT_CKPT = sorted(Path(SFT_OUTPUT).glob('*.pt'), key=lambda p: p.stat().st_mtime)[-1]
print('SFT checkpoint:', SFT_CKPT)

In [ ]:
# ── §4.3 — GRPO ──────────────────────────────────────────────────────────────
import subprocess, sys, os
from pathlib import Path

GRPO_PATH = Path(GRPO_DIR)
GRPO_PATH.mkdir(parents=True, exist_ok=True)
existing = sorted(GRPO_PATH.glob('*.pt'), key=lambda p: p.stat().st_mtime)
if existing:
    resume = existing[-1]
elif (GRPO_PATH / 'grpo_resume_hf.pt').exists():
    resume = GRPO_PATH / 'grpo_resume_hf.pt'
else:
    resume = SFT_CKPT
print('GRPO from:', resume)

_gcmd = [
    sys.executable, '-m', 'train.grpo',
    '--traces', 'data/train_full.jsonl',
    '--checkpoint', str(resume),
    '--output-dir', GRPO_DIR,
    '--model-size', '700m',
    '--group-size', str(GRPO_GROUP),
    '--lr', str(GRPO_LR),
    '--max-steps', str(GRPO_STEPS),
    '--kl-coeff', str(GRPO_KL),
    '--max-new-tokens', '1024',
    '--temperature', '0.8',
    '--device', DEVICE,
]
subprocess.run(_gcmd, check=True, env={**os.environ, 'HF_TOKEN': HF_TOKEN or '', 'HF_MODEL_REPO': HF_MODEL_REPO, 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'})

GRPO_CKPT = sorted(GRPO_PATH.glob('*.pt'), key=lambda p: p.stat().st_mtime)[-1]
print('GRPO checkpoint:', GRPO_CKPT)

In [ ]:
# ── OPTIONAL C: Self-improvement (PAPER §4.4) ────────────────────────────────
import subprocess, sys, os
from pathlib import Path

if SELF_IMPROVE_ITERS <= 0:
    print('SKIP self-improve (SELF_IMPROVE_ITERS=0). Full teacher regen: run scripts/pipeline_96.sh locally.')
else:
    grpo_files = sorted(Path(GRPO_DIR).glob('*.pt'), key=lambda p: p.stat().st_mtime)
    current = grpo_files[-1] if grpo_files else sorted(Path(SFT_OUTPUT).glob('*.pt'), key=lambda p: p.stat().st_mtime)[-1]
    for it in range(SELF_IMPROVE_ITERS):
        print(f'=== Self-improve iter {it+1}/{SELF_IMPROVE_ITERS} ===')
        si_sft = f'checkpoints/self_improve/sft_iter{it}'
        si_grpo = f'checkpoints/self_improve/grpo_iter{it}'
        Path(si_sft).mkdir(parents=True, exist_ok=True)
        Path(si_grpo).mkdir(parents=True, exist_ok=True)
        subprocess.run([
            sys.executable, '-m', 'train.sft_reasoning',
            '--traces', 'data/train_full.jsonl',
            '--output-dir', si_sft,
            '--model-size', '700m',
            '--epochs', '2',
            '--lr', '1e-4',
            '--batch-size', '1', '--grad-accum', '16', '--max-seq-len', str(SFT_MAX_SEQ),
            '--init-checkpoint', str(current),
            '--device', DEVICE,
        ], check=True, env=os.environ)
        sft_i = sorted(Path(si_sft).glob('*.pt'), key=lambda p: p.stat().st_mtime)[-1]
        subprocess.run([
            sys.executable, '-m', 'train.grpo',
            '--traces', 'data/train_full.jsonl',
            '--checkpoint', str(sft_i),
            '--output-dir', si_grpo,
            '--model-size', '700m',
            '--group-size', str(GRPO_GROUP),
            '--lr', '3e-6',
            '--max-steps', '2000',
            '--kl-coeff', str(max(0.01, 0.04 / (it + 1))),
            '--device', DEVICE,
        ], check=True, env=os.environ)
        current = sorted(Path(si_grpo).glob('*.pt'), key=lambda p: p.stat().st_mtime)[-1]
    print('Self-improve final checkpoint:', current)


In [ ]:
# ── OPTIONAL D: Verifier (PAPER abstract / §5.2) ───────────────────────────
import subprocess, sys, os
from pathlib import Path

if not RUN_VERIFIER:
    print('SKIP verifier')
else:
    Path(VERIFIER_DIR).mkdir(parents=True, exist_ok=True)
    subprocess.run([
        sys.executable, '-m', 'train.train_verifier',
        '--traces', 'data/train_full.jsonl',
        '--output-dir', VERIFIER_DIR,
        '--epochs', str(VERIFIER_EPOCHS),
        '--lr', '1e-4',
        '--device', DEVICE,
    ], check=True, env=os.environ)

In [ ]:
# ── Export + upload to Hugging Face ─────────────────────────────────────────
import subprocess, sys, os
from pathlib import Path

paths = []
for root in ('checkpoints/self_improve', 'checkpoints/grpo', 'checkpoints/sft_v2'):
    rp = Path(root)
    if rp.exists():
        paths.extend(rp.rglob('*.pt'))
if not paths:
    raise RuntimeError('No checkpoints found')
best = max(paths, key=lambda p: p.stat().st_mtime)
print('Export:', best)

subprocess.run([
    sys.executable, 'scripts/export_model.py',
    '--checkpoint', str(best),
    '--version', EXPORT_VERSION,
], check=True)

if PUSH_TO_HF and HF_TOKEN:
    subprocess.run([
        sys.executable, 'scripts/export_model.py',
        '--checkpoint', str(best),
        '--version', EXPORT_VERSION,
        '--push', '--repo', HF_MODEL_REPO,
    ], check=True, env={**os.environ, 'HF_TOKEN': HF_TOKEN})
    print('Uploaded to', HF_MODEL_REPO)
else:
    print('Skip push (set PUSH_TO_HF / HF_TOKEN)')


In [ ]:
# ── Optional: quick HumanEval sample (pass@k) ────────────────────────────────
import subprocess, sys, os
from pathlib import Path

cands = list(Path('checkpoints').rglob('*.pt'))
if cands and Path('scripts/eval_humaneval.py').exists():
    best = max(cands, key=lambda p: p.stat().st_mtime)
    subprocess.run([
        sys.executable, 'scripts/eval_humaneval.py',
        '--checkpoint', str(best),
        '--n-problems', '20',
        '--n-samples', '4',
    ], env=os.environ)
else:
    print('Skip eval (no checkpoint or script)')
